# Visual Autoregressive Models

推荐你读 https://arxiv.org/abs/2404.02905 Visual Autoregressive Modeling: Scalable Image Generation via Next-Scale Prediction 这是 VAR 原文，2024 NeurIPS Best Paper Award。

最近 GPT-Image 2.0 非常火爆，其性能远超了过去的图像生成巨头 Gemini Nano Banana Pro。关于其技术细节中，最重要的一条就是完全放弃了过去 OpenAI DALL-E 的扩散模型路线，全面转向 VAR。这证明了 VAR 已经可以达到甚至超越扩散模型的高度。

值得注意，我们正在越来越接近最前沿的技术路线。

# 基本逻辑

实际上 Autoregressive Generative Models 的历史并不短。VAR 之前的范式是 AR。AR 的核心思想是模仿大语言模型将图片像素展开为一维，然后做自回归预测下一像素 Token 的任务。这种方式非常早期且效果不好。下面这张图详细展示了 AR 与 VAR 的范式区别。

<img src="./assets/AR.png" width="800" height="330">

AR 假设图像是从左上角开始生成的，这完全不符合逻辑且破坏了图像的空间关系。并且生成一张图像需要预测巨量的 Token，推理速度低下。VAR 则完全打破这一逻辑，提出不再逐个像素预测，而是逐级分辨率预测。这是说模型先生成像素分辨率极低的图像，再根据已有的图像生成下一级像素分辨率更高的图像，循环这个过程到推理结束。

以上这个自回归过程意味着输入维度与输出维度一直在变化，这种变化的处理主要依赖 Transformer 不考虑序列长度关注全体上下文的强大特性。关于输入维度的变化，逐级拉直为一维向量配合位置编码是自然的想法。关于输出维度的变化，作者则提出先通过上一尺度的双线性插值上采样得到预计的符合输出维度的向量，然后同样拉直配合位置编码交给 Transformer 处理。我们后续会详谈这种有趣的架构如何实现。

类似大语言模型，VAR 也有自己的词表与 Tokenizer。或者换句话说，VAR 是一种极其特殊的 Latent "Diffusion" Model，也许我们可以发明一个词叫 Latent Autoregressive Model。

## Tokenization

我们通过 Vector Quantized-VAE Encoder 先将高维像素空间中原始图像 $im$ 映射到低维潜空间 $f \in \mathbb{R}^{h \times w \times C}$ ，而在低维潜空间上我们预设一个可学习的 Codebook $Z \in \mathbb{R}^{V \times C}$，通过将 $f$ 的每个通道方向长度为 $C$ 向量与 Codebook 每个长度为 $C$ 的词表向量进行对比，选出最靠近的向量取出其 ID。最终在对 $f$ 中每个位置都做过如上操作后得到 $q \in [V]^{h \times w}$。

换句话说 $$f = \mathcal{E}(im), \quad q = \mathcal{Q}(f)$$
此处 $im$ 代表原始图像，$\mathcal{E}(\cdot)$ 代表解码器，$\mathcal{Q}(\cdot)$ 代表量化子 (Codebook)。

更具体的，量化过程 $q = \mathcal{Q}(f)$ 将每个向量 $f^{(i,j)}$ 映射到其在 Euclidean 距离 (L2 范数) 意义下最近代码的代码索引 $q^{(i,j)}$ $$q^{(i,j)} = \left( \arg\min_{v \in [V]} \| \text{lookup}(Z, v) - f^{(i,j)} \|_2 \right) \in [V]$$
其中 $\text{lookup}(Z, v)$ 表示从代码本 $Z$ 中取第 $v$ 个向量。

反向的，在得到模型预测的 Token 序列结果之后，我们可以先通过 Codebook 还原为潜空间中表征，再通过 Decoder $\mathcal{D}(\cdot)$ 解码到原像素空间，作为最终生成结果。

在原文中，VQVAE 下采样率是 $H/h = W/w = 16 \times$。以原代码仓库局里，原像素空间 $256 \times 256 \times 3$ 经过 $\mathcal{E}$ 编码之后变为潜空间 $16 \times 16 \times 32$ 中的元素，所以总压缩率是 $1/24 \times$。

但是以上还不是 VQ-VAE 的全体。最后的，作者提出还需要一系列的卷积网络 $\{\phi_k\}_{k=1}^K$。对于每个 $\phi_k$，其接收张量维度与输出张量维度都是 $h \times w$。所以这是一个丰富语义的网络，我们需要他来建立 Token Map 到原 Feature Map 的直接联系，让线性插值结果没那么生硬。

我们在这里比较困难介绍卷积网络族 $\{\phi_k\}_{k=1}^K$ 被如何使用，因为我需要先解释 VAR 的生成方式。在下一小节你会看到 $\{\phi_k\}_{k=1}^K$ 如何参与生成。更多的，我们在最后还会介绍 VQVAE 如何联合训练这 $4$ 部分神经网络 $\mathcal{E},\mathcal{D},\mathcal{Q},\{\phi_k\}_{k=1}^K$。

## 生成方式

VAR 的生成方式是自回归，并且不是逐像素级别而是尺度级别。我们详细解释。

VAR 认为认为图像的最小自回归单元应该是一整张 Token Map。

我们称原始像素空间图像经过 Encoder 编码后的潜空间表征为 Feature Map $f \in \mathbb{R}^{h \times w \times C}$，那么我们可以将一张 Feature Map 量化为 $K$ 多量级的 Token Maps $$(r_1,r_2,...,r_K)$$ 其中每张 Token Map 拥有递增的维度 $h_k \times w_k$，并且我们要求最后一张 Token Map 符合原 Feature Map 维度，也就是说 $h_K = h$ 且 $w_k = w$。

现在我们可以定义整个自回归过程 $$p(r_1, r_2, \dots, r_K) = \prod_{k=1}^K p(r_k \mid r_1, r_2, \dots, r_{k-1})$$
这意味，在预测第 $k$ 个 Token Map 时，模型仅仅依赖 $(r_1,r_2,...,r_{k-1})$ 并且层级内所有的 $h_k \times w_k$ 个 Token 是同时生成的。

这样的生成方式是更加符合人类直觉的，我们从粗到细逐级勾勒图像。这样逐级生成计算效率也极高，将传统自回归生成的速度从像素线性级别提升到指数级别。更多的，由于是自回归式生成，KV Cache 现有的成熟技术可以完全迁移。

下面我们来说如何做 Feature Map 与 Token Maps 的 Tokenization。

### 图像到 Token Maps 序列的 Tokenization

所以，对于一张图像 $im$，如何使用我们已经训练好的 $\mathcal{E},\mathcal{D},\mathcal{Q}, \{\phi_k\}_{k=1}^K$ 来做 Tokenization？换句话说，我们希望用一张图片得到一个 Token Maps 序列，这样我们可以做 VAR 自回归的训练；同时我们希望一个 Token Maps 序列可以得到一张真实图片，这样 VAR 可以完成自洽的自回归推理。

直觉上我们可以直接通过简单上采样或者下采样完成这件事，然而作者提出一种残差式的 Tokenization，并且证明了其比简单采样更有效。我们详细说。

我们先来说编码。假设我们有已经训练好的 $\mathcal{E},\mathcal{D},\mathcal{Q}$，一张原始真实图片 $im$，已经设定的总步数 $K$ 与各个 Token Maps 维度 $(h_k,w_k)_{k=1}^{K}$，空的序列 $R = [ ]$，以及一系列已经训练好的卷积网络 $\{\phi_k\}_{k=1}^K$。

现在我们开始。首先得到 Feature Map $f = \mathcal{E}(im)$，初始步数 $k =1$。对于目前的步数 $k$，做下面几件事

通过自适应平均池化层将 $f$ 下采样到维度 $h_k \times w_k$，然后将池化结果由量化子量化到 Token Map $r_k$。

现在将 $r_k$ 塞入 $R$ 队尾。

再做 $\text{lookup}$ 操作，通过 $r_k$ 每个位置对应的 Token ID 找回原 Codebook 中向量得到 $z_k$。

$z_k$ 的维度是 $\mathbb{R}^{h_k \times w_k \times C}$，我们通过双线性插值上采样到 Feature Map 维度 $\mathbb{R}^{h_K \times w_K \times C}$。这里双线性插值指的是，对于 $h \times k$ 每个位置，将其映射到 $h_k \times w_k$ 中的非整数坐标，再取周围像素做距离上加权平均。更详细的 $$z_k^{new}(x, y) = \sum_{i,j \in h_k \times w_k} w_{i,j} \cdot z_{k}^{(i,j)}$$

最后我们计算 $f =f - \phi_k(z_k)$ 将其作为下一次循环使用的 Feature Map。进入下一个步数 $k+1$。

最终，我们会得到长度为 $K$ 的 Token Maps 序列 $R = (r_1,r_2,...,r_K)$，对应原始图像 $im$。

完整算法如下。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1: } \text{Multi-scale VQVAE Encoding} \\
\hline
1 \enspace \textbf{Inputs: } \text{raw image } im; \\
2 \enspace \textbf{Hyperparameters: } \text{steps } K, \text{resolutions } (h_k, w_k)_{k=1}^K; \\
3 \enspace f = \mathcal{E}(im), R = []; \\
4 \enspace \textbf{for } k = 1, \dots, K \textbf{ do} \\
5 \enspace \quad r_k = \mathcal{Q}(\text{interpolate}(f, h_k, w_k)); \\
6 \enspace \quad R = \text{queue\_push}(R, r_k); \\
7 \enspace \quad z_k = \text{lookup}(Z, r_k); \\
8 \enspace \quad z_k = \text{interpolate}(z_k, h_K, w_K); \\
9 \enspace \quad f = f - \phi_k(z_k); \\
10 \enspace \textbf{Return: } \text{multi-scale tokens } R; \\
\hline
\end{array}$$

现在我们来说，反向的，Token Maps 序列转回原图像。假设我们已有完整的 Token Maps 序列 $R = (r_1,r_2,...,r_K)$ 与空的复原图像 $\hat{f} = 0 \in \mathbb{R}^{h \times w \times C}$。

初始步数 $k = 1$，对目前的步数 $k$ 做下面几件事

取出 $R$ 中的 $r_k$，$\text{lookup}$ 原 Codebook 得到 $z_k \in \mathbb{R}^{h_k \times w_k \times C}$。

做双线性插值上采样到维度 $h \times w$。最后计算 $\hat{f} = \hat{f} + \phi_k(z_k)$ 将其作为下一次循环的复原 Feature Map。


所有循环结束之后，使用 $\mathcal{D}$ 得到复原图像 $\hat{im} = \mathcal{D}(\hat{f})$。

完整算法如下。

$$\begin{array}{l}
\hline
\textbf{Algorithm 2: } \text{Multi-scale VQVAE Reconstruction} \\
\hline
1 \enspace \textbf{Inputs: } \text{multi-scale token maps } R; \\
2 \enspace \textbf{Hyperparameters: } \text{steps } K, \text{resolutions } (h_k, w_k)_{k=1}^K; \\
3 \enspace \hat{f} = 0; \\
4 \enspace \textbf{for } k = 1, \dots, K \textbf{ do} \\
5 \enspace \quad r_k = \text{queue\_pop}(R); \\
6 \enspace \quad z_k = \text{lookup}(Z, r_k); \\
7 \enspace \quad z_k = \text{interpolate}(z_k, h_K, w_K); \\
8 \enspace \quad \hat{f} = \hat{f} + \phi_k(z_k); \\
9 \enspace \hat{im} = \mathcal{D}(\hat{f}); \\
10 \enspace \textbf{Return: } \text{reconstructed image } \hat{im}; \\
\hline
\end{array}$$

所以可以看到，卷积网络族 $\{\phi_k\}_{k=1}^K$ 实际上被用来平滑双线性插值上采样之后的生硬图像。

# 训练与推理

在 VAR 原论文中，作者几乎完全没有展示他们的模型架构与训练推理方式，但是提到架构与 GPT-2 极其相似。

鉴于这本教程中还完全没介绍过 GPT-2 式的生成，我们讲讲 VAR 的架构。这是代码仓库 https://github.com/FoundationVision/VAR 我非常推荐按照 REDAME.md 所说的运行一遍 `demo_sample.ipynb` 文件。原因是，某种意义上，非常有趣。从下载权重到实际运行用不了十分钟，你可以快速发现一些目前图像生成领域的已有成果与问题。

下面这张图是我使用 VAR d16 生成的，主模型参数量大小是 $310M$。在 ImageNet 数据集中这个标签名是 Cloaks。非常扭曲且诡异。

<img src="./assets/VARgene.png" width="1000" height="280">

但是这并不代表这个模型很糟糕。相反的，这个 $310M$ 参数大小的图像生成模型已经很强大了。同参数量级的其他模型甚至表现更差。如果尝试生成老鹰或者老虎机这类极其固定坚硬的图像，效果会很好。实际上斗篷这类柔软物体以及人体的生成困难是目前待解决的主流问题。我仍然建议你亲自去尝试生成。

总之，现在我们来看看这个网络到底是怎么一回事。下面这张图给出了 VAR 使用的主网络架构。

<img src="./assets/VAR_arch.png" width="500" height="620">

这个架构，某种意义上极其类似 DiT。Shared AdaLin 实际上就是共享的 adaLN-Zero 版本。我指原版 adaLN-Zero 中每个 Block 都会重新为条件编码重新投影到放缩参数与偏移参数，但是 Shared AdaLin 却在 Block 之外就将条件编码统一投影，所有 Block 共享这些系数，只是 Block 内部还会为系数加上一个独享的可学习偏移量。这大大减少了计算负担。

其他部分是熟悉的。关于最下方的输入编码，我们记模型内部隐藏层维度为 $D$，那么这一步核心目的就是将原本的潜空间向量 $\mathbb{R}^{1 \times 1 \times C}$ 序列转化到 $\mathbb{R}^{1 \times 1 \times D}$ 序列并且赋予对应的绝对位置编码。

然而我在这里详细解释输入如何运作恐怕很困难，我必须先讲述推理再详谈这一部分。下一小节我为你解释。

## 推理

我们详谈 VAR 的自回归式推理。

在 $k=0$ 时，我们什么也没有。此时我们实际上由一个特殊的起始符 `<sos>` 来发起第一次生成，这个向量是通过直接取出 Class Label 的 Embdeding 来实现的 (ImageNet 数据集中 $0 \sim 999$ 每个数字代表一种图像类别，无条件生成会对应 $1000$)。更多的，原代码仓库还为首个位置单独设计了可学习的起始位置编码 `pos_start`，用以表示这是首个生成位置。

最后是绝对位置编码与尺度编码。我们需要对每个位置编码两次，是因为我们需要表明这是第一个尺度也是第一个位置。

所以第一次进入模型的向量实际上是 `<sos>` + `pos_start` + `lvl_pos` 相加。他们的维度都是 $D$ 因此不需要额外的变换。

与此同时，Class Label 的 Embdeding 还会进入条件编码区域，为整个过程指导 (Shared AdaLin)。

最终在输出位置，模型会从大小为 $V$ 的词表中选择概率最高的或者 Top-k 采样，最终选择一个 Token Map。我们得到了 $r_1$。

$k=0$ 是最特殊的生成，接下来的过程可以统一叙述。假设 $k \ge 1$，我们已经生成Token Maps 序列 $R = (r_1,r_2,...,r_k)$。

首先，我们查表 Codebook 将每个 $r_n$ 转为 $z_n \in \mathbb{R}^{h_n \times w_n \times C}$，然后展开为一维向量组 $(z_n^{(1,1)}, z_n^{(1,2)},...,z_n^{(h_n,w_n)}) \in \mathbb{R}^{(h_n \times w_n) \times C}$。

所以总的序列是 $(z_1^{(1,1)}, z_2^{(1,1)}, z_2^{(1,2)}, z_2^{(2,1)}, z_2^{(2,2)}, z_3^{(1,1)},...,z_k^{(1,1)},..,z_k^{(h_k,w_k)})$。

现在我们开始着手准备下一个 Token Map $r_{k+1}$ 的生成。

我们先构造 $(z_{k+1}^{(1,1)},...,z_{k+1}^{(h_{k+1},w_{k+1})})$ 的雏形，这需要用到 $(z_k^{(1,1)}, z_k^{(1,2)},...,z_k^{(h_k,w_k)})$ 。将 $(z_k^{(1,1)}, z_k^{(1,2)},...,z_k^{(h_k,w_k)})$ 通过双线性插值上采样到 Feature Map 空间 $\mathbb{R}^{h \times w \times C}$。我们记新的上采样结果为 $z_k^{new} \in \mathbb{R}^{h \times w \times C}$。

顺便的，在这个过程中我们开始着手准备 $\hat{im}$ 的生成。利用卷积网络族 $\{\phi_k\}_{k=1}^K$ 计算 $\hat{f} = \hat{f} + \phi_k(z_k^{new})$。

我们再对 $\hat{f}$ 做下采样到 $\mathbb{R}^{h_{k+1} \times w_{k+1} \times C}$，这就是 $(z_{k+1}^{(1,1)},...,z_{k+1}^{(h_{k+1},w_{k+1})})$ 的雏形。我们记这个雏形为 $(q_{k+1}^{(1,1)},...,q_{k+1}^{(h_{k+1},w_{k+1})})$。

将这个雏形拼接进入 $(z_1^{(1,1)}, z_2^{(1,1)},..,z_k^{(h_k,w_k)})$ 得到 $(z_1^{(1,1)}, z_2^{(1,1)},..,z_k^{(h_k,w_k)},q_{k+1}^{(1,1)},...,q_{k+1}^{(h_{k+1},w_{k+1})})$。

我们开始进入我们的主网络。首先上面这组序列的通道维度是 $C$，我们通过 Word Embedding 层将其投影到通道维度为 $D$ 的隐藏层空间。随后我们开始进行两次绝对位置编码。先从尺度意义上为拥有同一下标的向量加上同一尺度的位置编码，其次根据每个向量在整体序列中的位置进行位置编码。

请注意以上是我为了方便理解叙述的，实际上我们根本不会让 $(z_1^{(1,1)}, z_2^{(1,1)},..,z_k^{(h_k,w_k)})$ 进入 Word Embedding 以及参与位置编码，而是直接取出 KV-Cache 中前几轮已经计算完毕的编码结果。这样可以大幅优化计算量。最终进入主网络的只有 $(q_{k+1}^{(1,1)},...,q_{k+1}^{(h_{k+1},w_{k+1})})$ 的 Word Embedding 以及位置编码结果。

更多的，原代码仓库中选择了简单直接的可学习位置编码而不是正余弦位置编码。

张量开始流动。直到 Multi-Head Self-Attention 模块前，我们做常规的模块处理操作。但是在注意力模块，我们取出 KV-Cache 中缓存的 $(K_{history}, V_{history})$ 拼接新的注意力矩阵 $(K_{new}, V_{new})$，再与新的注意力矩阵 $Q_{new}$ 做注意力计算。最终取出最后的 $h_{k+1} \times w_{k+1}$ 个注意力结果作为这一模块的输出。这个过程完全等价对整个完整历史序列做自注意力。

现在这个长度为 $h_{k+1} \times w_{k+1}$ 且通道数为 $D$ 的向量序列经过层层变换，最终来到输出层。在这里，我们先通过一个线性层将其通道数变换为词表大小 $V$，这意味着通道数的暴增。

最终产生 $r_{k+1}$ 的方式非常简单，我们对每条向量做 Softmax 后 Top-k 采样得到一个数字 ID，这就是 $r_{k+1}$ 每个位置在词表中的 ID。

所以当 $r_K$ 产生时，复原特征图 $\hat{f}$ 实际上也接近完整。但是请不要遗漏最后一步，就是 $r_K$ 产生之后收尾性的查表得到 $z_K$，计算 $\hat{f}= \hat{f} + \phi_K(z_K)$。注意 $z_K$ 无需上采样因为他的维度和 $\hat{f}$ 是一样的。

最后，$\hat{im} = \mathcal{D}(\hat{f})$。

## 训练

VAR 的训练分为两部分。首先第一阶段是训练我们所需的 VQVAE，包含 Encoder $\mathcal{E}$, Decoder $\mathcal{D}$, Quantizer $\mathcal{Q}$, 卷积网络族 $\{\phi_k\}_{k=1}^K$。第二阶段则是训练负责自回归生成的主网络。

### 训练 VQVAE

所以如何训练这个 VQVAE？我们实际上要联合训练 $4$ 个神经网络，这极其困难。我们绝对不会将 VQVAE 的训练交给自己完成，而是直接使用现有的类似 SD3 的 VAE 网络。在 LDM 章节我们已经见过联合训练 VAE 的难度，实际上 VQVAE 只会更困难。但是我还是愿意一说。

请不要忘记我们在生成方式中所说的 $Algorithm 1$ 与 $Algorithm 2$，他们实际上演示了整个训练流程。假设我们有待训练的已初始化 $\mathcal{E},\mathcal{D},\mathcal{Q}, \{\phi_k\}_{k=1}^K$ 以及真实数据集 $\{im\}$。

现在我们可以进行 $Algorithm 1$ 与 $Algorithm 2$，简而言之就是我们先由真实图像 $im$ 得到 Feature Map $f$，经过系列操作得到重构 Feature Map $\hat{f}$ 与重构图像 $\hat{im}$。我们不赘述如何进行。

定义损失函数
$$im \in \{im\},  \quad f = \mathcal{E}(im),  \quad \hat{f} = \text{Reconsruct}(f), \quad \hat{im} = \mathcal{D}(\hat{f})$$ $$\mathcal{L}  = \mathcal{L}_{Reconstrcut} + \alpha \mathcal{L}_{Codebook} + \beta \mathcal{L}_{Commitment}$$
一般取 $\alpha = 1, \beta = 0.25$。我们详细解释每一项，实际上损失函数形式极其相似我们在 LDM 中提到的 VAE 损失函数形式。

首先 $$ \mathcal{L}_{Reconstrcut} = \|im - \hat{im}\|_1 + \lambda \mathcal{L}_{LPIPS}(im, \hat{im}) + \lambda \mathcal{L}_{GAN} $$
这一项非常复杂，直觉上我们希望 $im$ 与 $\hat{im}$ 尽量接近，但是代价是引入 $LPIPS$ 度量网络与 GAN 网络 $Deter$，其中 GAN 网络也会参与联合训练。这意味着如果我们想要从头开始训练 VQVAE，我们需要训练整整 $5$ 个网络，这种难度完全不是个人或者小型实验室可以承担的。所以你现在知道我们完全不会自己训练 VQVAE 的原因。

其次的 $$\mathcal{L}_{Codebook} = \| sg[f] - \hat{f} \|_2^2$$
此处 $sg$ 指 $stopgrad$，换言之，我们将 $sg[f]$ 视为常数，希望 Codebook 查表产生的 $\hat{f}$ 能够尽量接近 $f$。

最后的 $$\mathcal{L}_{Commitment} = \| f - sg[\hat{f}]\| _2^2$$
这一项则是反过来，我们希望 $\mathcal{E}$ 输出尽量接近 Codebook 查表结果 $\mathcal{f}$。这其实很反直觉，但是合理。我们希望 $\mathcal{E}$ 承诺性地不产生巨大的映射偏差，否则会阻碍 Codebook 的训练。

这里存在一个工程问题，那就是 Quantizer 的 $\text{lookup}$ 操作是无法传播梯度的。但是一个工程技巧是 Straight-Through Estimator。我们直接将反向传播中 $\hat{f}$ 的梯度传播给 $f$，原因是假设他们相似度很高。

在 ${\mathcal{E},\mathcal{Q},\mathcal{D}, \{\phi_k\}_{k=1}^K}$ 全部训练完毕之后，可以开始进行正式的 VAR 训练。如我们已经提到的，这一步我们一般直接采用成熟的预训练 VQVAE。

### 训练主网络

主网络的训练会比 VQVAE 的训练简单得多。我们做标准的自回归训练。

对于一张真实图片 $im$，我们采取 $Algorithm 1$ 将其转化为 Token Maps 序列 $R = (r_1,...,r_K)$，再查表 Codebook 建议其转化为 $(z_1,...,z_K)$，展开得到 $(z_1^{(1,1)}, z_2^{(1,1)}, z_2^{(1,2)},..,z_K^{(h_K,w_K)})$。

现在我们一次性计算组装所有雏形序列。对于尺度 $n$，我们使用 $(z_{n-1}^{(1,1)},...,z_{n-1}^{(h_{n-1},w_{n-1})})$ 通过我们已经讲述的采样方式得到雏形 $(q_{n}^{(1,1)},...,q_{n}^{(h_{n},w_{n})})$。记 $q_1 = $`<sos>` + `<pos_start>` 作为 $z_1^{(1,1)}$ 的雏形。

所以我们可以组装雏形列 $(q_1,q_2^{(1,1)}, ..., q_K^{(h_K,w_K)})$。

一个巨大便利的工程技巧是直接将整个雏形序列一次输入主模型而不是分次输入预测每个位置。原因是，首先 MLP 不关注序列维度向量的，因此一个向量输入与一个向量序列是同理的；其次是最重要的 Multi-Head Self-Attention 模块，我们需要保证 Attention 不可以在预测 $r_n$ 时偷看 $r_n$ 本身的结果，这需要 Next-Scale Mask。

在 LLM 中，这种因果掩码是严格的，预测每个词仅仅允许查看此词之前的所有序列。在 VAR 中这种掩码则是分块的进行的。

简而言之，我们记 $d = [1,2,2,2,2,3,3,3,3,3,3,3,3,3,4,...]$ 这个序列标记了每个位置的尺度序号。在 Multi-Head Self-Attention 计算中，我们允许且仅仅允许当注意力矩阵 $Q$ 的尺度序号大于等于注意力矩阵 $K$ 尺度序号时进行注意力运算，否则我们记注意力矩阵该位置结果为 $-\text{inf}$。

在标准 LLM 自回归中，第 $i$ 个词只能看到它之前的词。如果序列长度为 $L$，掩码矩阵 $M$ 为$$M = \begin{pmatrix} 
0 & -\infty & -\infty & \dots \\
0 & 0 & -\infty & \dots \\
0 & 0 & 0 & \dots \\
\vdots & \vdots & \vdots & \ddots 
\end{pmatrix}$$

在 VAR 中，这一掩码矩阵则是分块的$$M = \left(
\begin{array}{c|cccc|ccccccccc}
0 & -\infty & -\infty & -\infty & -\infty & -\infty & \dots \\ \hline
0 & 0 & 0 & 0 & 0 & -\infty & \dots \\
0 & 0 & 0 & 0 & 0 & -\infty & \dots \\
0 & 0 & 0 & 0 & 0 & -\infty & \dots \\
0 & 0 & 0 & 0 & 0 & -\infty & \dots \\ \hline
0 & 0 & 0 & 0 & 0 & 0 & \dots & 0 \\
\vdots & \vdots & \vdots & \vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & 0 & 0 & 0 & 0 & \dots & 0 
\end{array}
\right)$$
这意味着叠加这一掩码矩阵之后，原注意力矩阵仅仅只有左下方是正常的注意力权重，而在 Softmax 之后，右上方会全部变成 $0$。

换言之
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}} + M\right)V$$
实际上这完全等价在预测 $r_n$ 时输入 $(r_1,..,r_{n-1})$，这一技巧促成了训练速度巨大的提升。

在最后输出头时，我们将每个序列向量的通道数投影到词表大小 $V$。最后我们的损失函数定义为多类型交叉熵损失
$$L_{total} = \sum_{k=1}^{K} w_k \cdot L_{ce}(p_k, y_k)$$
这表示第 $n$ 尺度的预测全部共用同一权重函数 $w_n$。或者我们展开
$$L_{total} = \sum_{k=1}^{K} w_k \sum_{m=1}^{h_k \times w_k} (-\log p_{correct}^{(m)})$$

在遍历一个 Batch 的真实图像 $im$ 之后，我们计算平均损失反向传播更新梯度。

我们基本说完了。

# Scaling Law

最后我们简短说说 VAR 的 Scaling Law。VAR 的性能根据参数是呈 $1/3$ 次方放大的，换句话说，最终停训 $Loss \propto N^{-\alpha}$，其中 $\alpha$ 是模型深度 $d$。

更多的，作者给出了模型宽度与深度关系 $w= 64d$，Multi-Head Self-Attention 头数 $h=d$。因此总参数量关系是 $$N(d) \approx 18dw^2 = 73728d^3$$
所以最终 $Loss$ 与总参数量关系是呈 $1/3$ 次方。

# 总结

自回归式的生成作为完全不同于 Diffusion 式的生成架构，其在某些指标上展现出了超越 Diffusion 的表现。如对数级别的推理步数与严格的 Scaling Law。不过最重要的还是自回归架构完全统一了文字生成与图像生成，这意味着模型可以更加原生地理解文字语义与图片语义。

著名的自回归式生成模型有 LlamaGen, Gemini Nano Banana 2 与 GPT Image-2.0。不过 Diffusion 的顶尖模型仍然保持巨大的竞争力。

我们注意到，VAR 与 SD3 潜空间模型实际上都是基于 VAE 实现的。此处存在一个 VAE Decoder 性能上限问题，就是无论潜空间模型如何强大，实际上最后一步都会受到 Decoder 转换到真实空间的性能限制。近来随着潜空间模型越来越强大而 VAE 逐渐遇到瓶颈，如我们提到的多神经网络训练的巨大困难，我们开始重新思考能否在原像素空间训练生成模型。